# Excel Extraction — Assigning a Specific Agent

Demonstrates two ways to pin a job to a specific agent (or agent pool) when submitting
an `@istari:extract` job against an Excel `.xlsx` file using the `open_spreadsheet` tool.

| Option | Approach | When to use |
|--------|----------|-------------|
| **1** | Drop to `platform.client.add_job()` directly | One-off, no library changes needed |
| **2** | Extend `JobDefinition` with agent fields | Repeated use, keeps the fluent style |

### Prerequisites

- An **Istari Digital Platform account** and a **Personal Access Token**.
- An agent with an on-premises module loaded (e.g. `@istari:dassault_cameo`).

### Credentials

Create a `.env` file next to this notebook:

```
ISTARI_REGISTRY_URL=https://...paste your platform's registry URL here...
ISTARI_PERSONAL_ACCESS_TOKEN=...paste your token here...
```

## 1 · Connect

In [18]:
from pathlib import Path

from istari_fluent import IstariPlatform, JobDefinition, JobView

platform = IstariPlatform.from_env()

report = platform.client.readiness_check()
assert report.healthy, f"Platform reports unhealthy: {report}"

print(platform)

2026-05-18 17:11:17 -  istari_digital_client.compatibility:process_response_headers:81 - WARNING - SDK is incompatible with Istari Registry v10.15.2 (affected APIs: v2_main:Systems). Please update your SDK to match the server version.
2026-05-18 17:11:17 -  istari_digital_client.compatibility:process_response_headers:81 - WARNING - SDK is incompatible with Istari Registry v10.15.2 (affected APIs: v2_main:Systems). Please update your SDK to match the server version.
2026-05-18 17:11:17 -  istari_digital_client.compatibility:process_response_headers:81 - WARNING - SDK is incompatible with Istari Registry v10.15.2 (affected APIs: v2_main:Systems). Please update your SDK to match the server version.


IstariPlatform connected to https://fileservice-v2.demo.istari.app


## 2 · Configure

In [ ]:
XLSX_PATH       = Path.cwd() / "Group3-UAS-Requirements.xlsx"
DISPLAY_NAME    = "Group3-UAS-Requirements.xlsx"
EXTERNAL_ID     = "agent-assignment-uas-requirements-demo"

# Set this to the agent ID you want to target (see Section 3 below).
# Leave as None to skip the agent-assigned jobs and just browse available agents.
TARGET_AGENT_ID = "f380e804-f63b-451c-b272-45c728cf5556"  # e.g. "agt-abc123"

assert XLSX_PATH.exists(), f"File not found: {XLSX_PATH}"
print(f"Input file:  {XLSX_PATH}")
print(f"Agent ID:    {TARGET_AGENT_ID or '(not set — browse agents in Section 3 first)'}")

Input file:  /Users/craighahn/Documents/GitHub/cookbook/istari-digital-client-cookbook/samples/Group3-UAS-Requirements.xlsx
Agent ID:    6882376a-b5d4-469a-8817-0ff66cd763f1


## 3 · Browse available agents

List all agents that have the `open_spreadsheet` module loaded and are currently active.

> **Note:** Auto-select may route jobs to a cloud-hosted execution environment
> (`agent_id` will be `None` on the completed job) rather than any registered agent.
> If your auto-selected jobs show `agent_id: None`, explicit agent assignment targets
> on-premises agents only and will behave differently from auto-select. You can check
> which agent (if any) ran a previous job with:
> ```python
> job = platform.client.get_job(job_id="<id>")
> print(job.agent_id)   # None = cloud execution, a UUID = specific agent
> ```

In [20]:
agents = platform.client.list_agents(
    module_name="@istari:open_spreadsheet",
    size=25,
)

print(f"Agents with @istari:open_spreadsheet: {agents.total}\n")
print(f"{'AGENT ID':<40} {'DISPLAY NAME':<30} {'OS':<20} {'STATUS'}")
print("-" * 110)
for a in agents.items:
    display_name = a.display_name.display_name if a.display_name else ""
    status       = a.status.name if a.status else "Unknown"
    print(f"  {a.id:<38} {display_name:<30} {(a.host_os or ''):<20} {status}")

# Agents with status Idle are ready to accept work.
# Copy an ID above into TARGET_AGENT_ID in the config cell.
# If all your successful auto-selected jobs show agent_id=None (see note above),
# explicit assignment to these agents uses a different execution path.

Agents with @istari:open_spreadsheet: 347

AGENT ID                                 DISPLAY NAME                   OS                   STATUS
--------------------------------------------------------------------------------------------------------------
  a98352a0-ef38-47d7-ab56-a6a8aa9ea38b   gentle-aragorn-5873            RHEL 8               AgentStatusName.IDLE
  b36b4244-c5fe-4fd5-9811-065bc8b8f4be   heroic-celeborn-7221           RHEL 8               AgentStatusName.IDLE
  b19adb3e-5b11-428b-becb-69313543f5da   jaunty-yavanna-0262            RHEL 8               AgentStatusName.EXECUTINGJOB
  87b032ff-04a1-41a1-92b8-85b1aee9ec05   rested-quickbeam-2124          RHEL 8               AgentStatusName.IDLE
  52237890-344f-4848-990e-3c9d3c9bd7f7   nimble-ulmo-2790               RHEL 8               AgentStatusName.IDLE
  3a67d4b9-17b9-4fa5-aedd-a19397294d24   proper-gimli-8418              RHEL 8               AgentStatusName.EXECUTINGJOB
  65051b76-3c90-4526-aec5-307ec5ca8e9d   fierc

## 4 · Upload the spreadsheet as a Model resource

In [15]:
model = platform.upload_model(
    XLSX_PATH,
    external_id=EXTERNAL_ID,
    display_name=DISPLAY_NAME,
)
print(f"Uploaded model {model.id}")
print(model)

Uploaded model 76596e4a-cd0d-4aa2-a683-092ddb853562
Model('Group3-UAS-Requirements.xlsx', id=76596e4a-cd0d-4aa2-a683-092ddb853562, file=c5c4aaa1-66e0-4b3c-94a7-c3249f61ce7d, rev=f86b5b30-5b3e-4460-ad64-79869f8e7409)


## 5 · Option 1 — assign via `platform.client.add_job()`

Bypass `JobDefinition` entirely and call `add_job()` on the raw client.
This exposes every parameter the API supports, including `assigned_agent_id`
and `assigned_agent_pool_id`.

After submission, wrap the returned `Job` in a `JobView` to get `.wait()`,
`.get_products()`, and the rest of the fluent interface.

In [21]:
assert TARGET_AGENT_ID, "Set TARGET_AGENT_ID in the config cell before running this section."

job1_raw = platform.client.add_job(
    model_id=model.id,
    function="@istari:extract",
    tool_name="open_spreadsheet",
    assigned_agent_id=TARGET_AGENT_ID,
    # assigned_agent_pool_id="<pool-id>",  # alternative: assign to a pool
)

job1 = JobView(_job=job1_raw, _client=platform.client)
print(f"Submitted job {job1.id} assigned to agent {TARGET_AGENT_ID}; polling...")

job1.wait(
    timeout=600,
    on_poll=lambda j: print(f"  [{j.status}] id={j.id}"),
).on_success()

print(f"\nJob 1 finished: {job1.status}")
print(f"Executed by:    {job1._job.agent_id or '(cloud)'}")

products1 = job1.get_products()
print(f"Products: {[p.name for p in products1]}")

Submitted job 3cb87cc6-9c22-4342-a871-a070072254a3 assigned to agent 6882376a-b5d4-469a-8817-0ff66cd763f1; polling...
  [Pending] id=3cb87cc6-9c22-4342-a871-a070072254a3
  [Pending] id=3cb87cc6-9c22-4342-a871-a070072254a3
  [Pending] id=3cb87cc6-9c22-4342-a871-a070072254a3
  [Pending] id=3cb87cc6-9c22-4342-a871-a070072254a3
  [Pending] id=3cb87cc6-9c22-4342-a871-a070072254a3
  [Pending] id=3cb87cc6-9c22-4342-a871-a070072254a3
  [Pending] id=3cb87cc6-9c22-4342-a871-a070072254a3
  [Pending] id=3cb87cc6-9c22-4342-a871-a070072254a3
  [Pending] id=3cb87cc6-9c22-4342-a871-a070072254a3
  [Pending] id=3cb87cc6-9c22-4342-a871-a070072254a3
  [Pending] id=3cb87cc6-9c22-4342-a871-a070072254a3
  [Pending] id=3cb87cc6-9c22-4342-a871-a070072254a3
  [Pending] id=3cb87cc6-9c22-4342-a871-a070072254a3
  [Pending] id=3cb87cc6-9c22-4342-a871-a070072254a3
  [Pending] id=3cb87cc6-9c22-4342-a871-a070072254a3
  [Pending] id=3cb87cc6-9c22-4342-a871-a070072254a3
  [Pending] id=3cb87cc6-9c22-4342-a871-a070072254a

KeyboardInterrupt: 

## 6 · Option 2 — extend `JobDefinition`

For repeated use, subclass `JobDefinition` to add the agent fields, then
provide a small helper that calls `add_job()` and returns a `JobView`.
This keeps the submission call site as clean as the standard fluent style
without patching the library.

If you need this across multiple notebooks, move the class and helper into
a shared module (e.g. `istari_fluent/istari_utils.py`) and wire
`assigned_agent_id` through `_submit_job_impl`.

In [20]:
from pydantic import Field as PydanticField


class AgentJobDefinition(JobDefinition):
    """JobDefinition extended with agent/pool targeting."""
    assigned_agent_id: str | None = PydanticField(default=None)
    assigned_agent_pool_id: str | None = PydanticField(default=None)


def submit_job(platform: IstariPlatform, model_id: str, defn: AgentJobDefinition) -> JobView:
    """Submit an AgentJobDefinition and return a JobView ready to .wait()."""
    job_raw = platform.client.add_job(
        model_id=model_id,
        function=defn.function,
        tool_name=defn.tool_name,
        tool_version=defn.tool_version,
        operating_system=defn.operating_system,
        parameters=defn.build_parameters(),
        assigned_agent_id=defn.assigned_agent_id,
        assigned_agent_pool_id=defn.assigned_agent_pool_id,
    )
    return JobView(_job=job_raw, _client=platform.client)

In [21]:
assert TARGET_AGENT_ID, "Set TARGET_AGENT_ID in the config cell before running this section."

extract_on_agent = AgentJobDefinition(
    function="@istari:extract",
    tool_name="open_spreadsheet",
    assigned_agent_id=TARGET_AGENT_ID,
    # assigned_agent_pool_id="<pool-id>",
)

job2 = submit_job(platform, model.id, extract_on_agent)
print(f"Submitted job {job2.id} assigned to agent {TARGET_AGENT_ID}; polling...")

job2.wait(
    timeout=600,
    on_poll=lambda j: print(f"  [{j.status}] id={j.id}"),
).on_success()

print(f"\nJob 2 finished: {job2.status}")

products2 = job2.get_products()
print(f"Products: {[p.name for p in products2]}")

Submitted job e1543284-dfae-4249-9b42-9d8423aa5b17 assigned to agent 7a0653be-6f09-447d-857c-457df658abcb; polling...
  [Pending] id=e1543284-dfae-4249-9b42-9d8423aa5b17
  [Pending] id=e1543284-dfae-4249-9b42-9d8423aa5b17
  [Pending] id=e1543284-dfae-4249-9b42-9d8423aa5b17
  [Pending] id=e1543284-dfae-4249-9b42-9d8423aa5b17
  [Pending] id=e1543284-dfae-4249-9b42-9d8423aa5b17
  [Pending] id=e1543284-dfae-4249-9b42-9d8423aa5b17
  [Pending] id=e1543284-dfae-4249-9b42-9d8423aa5b17
  [Pending] id=e1543284-dfae-4249-9b42-9d8423aa5b17
  [Pending] id=e1543284-dfae-4249-9b42-9d8423aa5b17
  [Pending] id=e1543284-dfae-4249-9b42-9d8423aa5b17
  [Pending] id=e1543284-dfae-4249-9b42-9d8423aa5b17
  [Pending] id=e1543284-dfae-4249-9b42-9d8423aa5b17
  [Pending] id=e1543284-dfae-4249-9b42-9d8423aa5b17
  [Pending] id=e1543284-dfae-4249-9b42-9d8423aa5b17
  [Pending] id=e1543284-dfae-4249-9b42-9d8423aa5b17
  [Pending] id=e1543284-dfae-4249-9b42-9d8423aa5b17
  [Pending] id=e1543284-dfae-4249-9b42-9d8423aa5b1

KeyboardInterrupt: 

## Verify in the UI

1. **Jobs / Activity** — Both jobs should show `open_spreadsheet / @istari:extract`.
2. **Job detail** — Each job should show the assigned agent under its execution details.

## Optional · Archive the model

In [ ]:
model.archive()